In [2]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape of dataset:", df.shape)

df.head()

Shape of dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (7043, 21)

Columns:
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn             

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['TotalCharges'].dtype)

print(df['TotalCharges'].isnull().sum())

float64
11


In [5]:
#removing the missing values
df.dropna(inplace=True)

print("Shape after cleaning:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

Shape after cleaning: (7032, 21)

Missing Values:
0


In [6]:
#return all the distinct values present in the colunm
print("Contract Types:")
print(df["Contract"].unique())

print("\nPayment Methods:")
print(df["PaymentMethod"].unique())

print("\nInternet Service:")
print(df["InternetService"].unique())

print("\nChurn Values:")
print(df["Churn"].unique())

Contract Types:
['Month-to-month' 'One year' 'Two year']

Payment Methods:
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Internet Service:
['DSL' 'Fiber optic' 'No']

Churn Values:
['No' 'Yes']


In [7]:
#to predict the predicted_next_bill to calculate bill_drop
df["previous_month_bill"] = df["MonthlyCharges"]

df[["MonthlyCharges", "previous_month_bill"]].head()

,MonthlyCharges,previous_month_bill
0,29.85,29.85
1,56.95,56.95
2,53.85,53.85
3,42.30,42.30
4,70.70,70.70


In [8]:
import numpy as np
#same random values are generated every run.
np.random.seed(42)
#np.random.normal(mean, std, size)
#creates random values from a Normal Distribution
df["next_month_bill"] = (
    df["MonthlyCharges"]
    + np.random.normal(0, 5, len(df))
)

df[["MonthlyCharges", "next_month_bill"]].head()

,MonthlyCharges,next_month_bill
0,29.85,32.333571
1,56.95,56.258678
2,53.85,57.088443
3,42.30,49.915149
4,70.70,69.529233


In [9]:
X = df_model.drop(
    columns=[
        "next_month_bill"
    ]
)

y = df_model["next_month_bill"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

NameError: name 'df_model' is not defined

In [10]:
#textcolumns-num         to avoid dummy values
df_model = pd.get_dummies(df, drop_first=True)
# more col [
print("Shape:", df_model.shape)

Shape: (7032, 7064)


In [11]:
#remove cust id
df_ml = df.drop(columns=["customerID"])

In [12]:
df_model = pd.get_dummies(df_ml, drop_first=True)

print("Shape:", df_model.shape)

Shape: (7032, 33)


In [13]:
#input features x[tenure,MonthlyCharges,TotalCharges,Contract]
X = df_model.drop(columns=["next_month_bill"])
#traget variable
y = df_model["next_month_bill"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (7032, 32)
y Shape: (7032,)


In [15]:
#Used to split the data into trainng and testing data
from sklearn.model_selection import train_test_split
#80per train,20per test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5625, 32)
X_test : (1407, 32)
y_train: (5625,)
y_test : (1407,)


In [16]:
#to predict continuoous values
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
#Create Model
lr = LinearRegression()
#Train Model[X_train → Customer Features,y_train → Next Month Bill]
lr.fit(X_train, y_train)
#Make Predictions
y_pred_lr = lr.predict(X_test)
#Evaluate Model[predictions differ from actual bills by about ₹4.07.]
mae_lr = mean_absolute_error(y_test, y_pred_lr)
#prediction error is approximately ₹5.13
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression MAE :", mae_lr)
print("Linear Regression RMSE:", rmse_lr)

Linear Regression MAE : 4.073398629736228
Linear Regression RMSE: 5.129995653577779


In [17]:
#avoid overfitting by adding a penalty term
from sklearn.linear_model import Ridge
#Controls regularization strength.
ridge = Ridge(alpha=1.0)
#Model learns customer billing patterns
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)
#Calculate Errors
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print("Ridge MAE :", mae_ridge)
print("Ridge RMSE:", rmse_ridge) 

Ridge MAE : 4.073376754540258
Ridge RMSE: 5.12997671510485


In [18]:
from sklearn.ensemble import RandomForestRegressor
#Instead of using one decision tree, it uses many trees and combines their prediction
#Create Model[100 decision trees,Each tree predicts the bill amount.]
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest MAE :", mae_rf)
print("Random Forest RMSE:", rmse_rf) 

Random Forest MAE : 4.323061821408158
Random Forest RMSE: 5.424668207857055


In [19]:
import xgboost
print(xgboost.__version__)

3.2.0


In [20]:
import xgboost
print(xgboost.__version__)

3.2.0


In [25]:
#5625 training records32 input features[80% → Training → 5625,20% → Testing  → 1407]
print(X_train.shape)

(5625, 32)


In [21]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
#Average prediction error ≈ ₹4.17
#Average prediction error ≈ ₹5.24
print("XGBoost MAE :", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)

XGBoost MAE : 4.165760800301891
XGBoost RMSE: 5.240582459209599


In [22]:
import joblib
#Ridge Regression usually more stable because of regularization.
joblib.dump(ridge, "../model/bill_prediction_model.pkl")
#model/└── bill_prediction_model.pkl
print("Model Saved Successfully")

Model Saved Successfully


In [23]:
df["bill_drop"] = (
    df["previous_month_bill"]
    - df["predicted_next_bill"]
)

df[[
    "previous_month_bill",
    "predicted_next_bill",
    "bill_drop"
]].head()

KeyError: 'predicted_next_bill'

In [24]:
best_model = ridge
#The trained Ridge model predicts the future bill amount for every customer.
df["predicted_next_bill"] = best_model.predict(X)

print(df[[
    "MonthlyCharges",
    "predicted_next_bill"
]].head())

   MonthlyCharges  predicted_next_bill
0           29.85            30.218566
1           56.95            56.584041
2           53.85            53.767008
3           42.30            42.339620
4           70.70            70.968752


In [25]:
#to check the predicted_next_bill present
print(df.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'previous_month_bill', 'next_month_bill', 'predicted_next_bill'],
      dtype='object')


In [26]:
#Calculate Bill Drop[billdrop=previousmonthbill-predictednextbill]
df["bill_drop"] = (
    df["previous_month_bill"]
    - df["predicted_next_bill"]
)

df[[
    "previous_month_bill",
    "predicted_next_bill",
    "bill_drop"
]].head()

,previous_month_bill,predicted_next_bill,bill_drop
0,29.85,30.218566,-0.368566
1,56.95,56.584041,0.365959
2,53.85,53.767008,0.082992
3,42.30,42.339620,-0.039620
4,70.70,70.968752,-0.268752


In [27]:
threshold = 5
#Runs the function on every row(apply()
#Check each bill_drop value and assign a label.
df["bill_drop_label"] = df["bill_drop"].apply(
    lambda x: "High Drop" if x > threshold else "Normal"
)

print(df["bill_drop_label"].value_counts())

bill_drop_label
Normal    7032
Name: count, dtype: int64


In [28]:
print(type(ridge))

<class 'sklearn.linear_model._ridge.Ridge'>


In [29]:
df["predicted_next_bill"] = ridge.predict(X)

df[[
    "MonthlyCharges",
    "predicted_next_bill"
]].head()

,MonthlyCharges,predicted_next_bill
0,29.85,30.218566
1,56.95,56.584041
2,53.85,53.767008
3,42.30,42.339620
4,70.70,70.968752


In [35]:
df["bill_drop"] = (
    df["previous_month_bill"]
    - df["predicted_next_bill"]
)

df[[
    "previous_month_bill",
    "predicted_next_bill",
    "bill_drop"
]].head()

,previous_month_bill,predicted_next_bill,bill_drop
0,29.85,30.218566,-0.368566
1,56.95,56.584041,0.365959
2,53.85,53.767008,0.082992
3,42.30,42.339620,-0.039620
4,70.70,70.968752,-0.268752


In [30]:
threshold = 5

df["bill_drop_label"] = df["bill_drop"].apply(
    lambda x: "High Drop"
    if x > threshold
    else "Normal"
)

print(df["bill_drop_label"].value_counts())

bill_drop_label
Normal    7032
Name: count, dtype: int64


In [31]:
print(df["bill_drop_label"].value_counts())

bill_drop_label
Normal    7032
Name: count, dtype: int64


In [32]:
#Check Bill Drop Statistics
print(df["bill_drop"].describe())

count    7032.000000
mean        0.007402
std         0.328765
min        -1.059533
25%        -0.203410
50%         0.015774
75%         0.226198
max         1.283986
Name: bill_drop, dtype: float64


In [33]:
#Dynamic Threshold[Find the value below which 90% of customers fall.]
threshold = df["bill_drop"].quantile(0.90)

print("Threshold:", threshold)

Threshold: 0.41845548895125617


In [34]:
#Label Customers
df["bill_drop_label"] = df["bill_drop"].apply(
    lambda x: "High Drop"
    if x >= threshold
    else "Normal"
)

print(df["bill_drop_label"].value_counts())

bill_drop_label
Normal       6328
High Drop     704
Name: count, dtype: int64


In [35]:
#recomendation of offer
def recommend_offer(row):

    if row["bill_drop_label"] == "High Drop":

        if row["InternetService"] == "Fiber optic":
            return "Premium Entertainment Bundle"

        elif row["Contract"] == "Month-to-month":
            return "Loyalty Discount Offer"

        elif row["OnlineSecurity"] == "No":
            return "Online Security Add-on"

        else:
            return "Personalized Retention Offer"

    else:
        return "No Offer Needed"


df["recommended_offer"] = df.apply(
    recommend_offer,
    axis=1
)

print(df["recommended_offer"].value_counts())

recommended_offer
No Offer Needed                 6328
Premium Entertainment Bundle     308
Personalized Retention Offer     159
Loyalty Discount Offer           138
Online Security Add-on            99
Name: count, dtype: int64


In [36]:
print(df["recommended_offer"].value_counts())

recommended_offer
No Offer Needed                 6328
Premium Entertainment Bundle     308
Personalized Retention Offer     159
Loyalty Discount Offer           138
Online Security Add-on            99
Name: count, dtype: int64


In [37]:
#Automated Email Flag
df["email_sent"] = df["bill_drop_label"].apply(
    lambda x: True if x == "High Drop" else False
)

print(df["email_sent"].value_counts())

email_sent
False    6328
True      704
Name: count, dtype: int64


In [38]:
#Customer Segmentation
df["customer_segment"] = df["bill_drop_label"].apply(
    lambda x: "Outreach Customer"
    if x == "High Drop"
    else "Regular Customer"
)

print(df["customer_segment"].value_counts())

customer_segment
Regular Customer     6328
Outreach Customer     704
Name: count, dtype: int64


In [39]:
#Enterprise Customer Identification[High Value Customers,Long-Term Customers,Revenue Important Customers]
df["enterprise_customer"] = (
    (df["MonthlyCharges"] > 80)
    | (df["tenure"] > 50)
)

print(df["enterprise_customer"].value_counts())

enterprise_customer
True     3646
False    3386
Name: count, dtype: int64


In [40]:
final_df = df[[
    "customerID",
    "MonthlyCharges",
    "predicted_next_bill",
    "bill_drop",
    "bill_drop_label",
    "recommended_offer",
    "email_sent",
    "customer_segment",
    "enterprise_customer"
]]

final_df.head()

,customerID,MonthlyCharges,predicted_next_bill,bill_drop,bill_drop_label,recommended_offer,email_sent,customer_segment,enterprise_customer
0,7590-VHVEG,29.85,30.218566,-0.368566,Normal,No Offer Needed,False,Regular Customer,False
1,5575-GNVDE,56.95,56.584041,0.365959,Normal,No Offer Needed,False,Regular Customer,False
2,3668-QPYBK,53.85,53.767008,0.082992,Normal,No Offer Needed,False,Regular Customer,False
3,7795-CFOCW,42.30,42.339620,-0.039620,Normal,No Offer Needed,False,Regular Customer,False
4,9237-HQITU,70.70,70.968752,-0.268752,Normal,No Offer Needed,False,Regular Customer,False


In [41]:
final_df.to_csv(
    "../outputs/final_customer_scorecard.csv",
    index=False
)

print("Final Customer Scorecard Exported Successfully")

OSError: Cannot save file into a non-existent directory: '..\outputs'

In [42]:
final_df.to_csv(
    "../outputs/final_customer_scorecard.csv",
    index=False
)

print("Final Customer Scorecard Exported Successfully")

Final Customer Scorecard Exported Successfully


In [43]:
import joblib

joblib.dump(ridge, "model/bill_prediction_model.pkl")

print("Model Saved Successfully")

FileNotFoundError: [Errno 2] No such file or directory: 'model/bill_prediction_model.pkl'

In [44]:
import os

print(os.getcwd())

C:\Users\frans\OneDrive\Desktop\Telecom_Bill_Prediction_System\notebook


In [45]:
import joblib

joblib.dump(
    ridge,
    "../model/bill_prediction_model.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [46]:
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return "<h1>Telecom Bill Prediction System</h1>"

if __name__ == "__main__":
    app.run(debug=True)

ModuleNotFoundError: No module named 'flask'